<a href="https://colab.research.google.com/github/2303a51343/RL/blob/main/RL_11_1343.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


gamma = 0.9
theta = 1e-8
normal_reward = -1
wall_reward = -1
goal_reward = 100

states = ["S0", "S1"]
actions = ["Up", "Down", "Left", "Right"]
all_states = ["S0", "S1", "G"]



grid = {
    (0, 0): "S0",
    (0, 1): "Wall",
    (1, 0): "S1",
    (1, 1): "G"
}

positions = {
    "S0": (0, 0),
    "S1": (1, 0),
    "G": (1, 1)
}

moves = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1)
}


def step(state, action):
    row, col = positions[state]
    dr, dc = moves[action]
    next_position = (row + dr, col + dc)

    cell = grid.get(next_position)

    # Wall or boundary: stay in the current state
    if cell is None or cell == "Wall":
        return state, wall_reward

    # Reaching the goal
    if cell == "G":
        return "G", goal_reward

    # Normal movement
    return cell, normal_reward


# Initial policy and state values
policy = {"S0": "Up", "S1": "Up"}
V = {state: 0.0 for state in all_states}

print("TRANSITION INFORMATION")
for state in states:
    for action in actions:
        next_state, reward = step(state, action)
        print(
            f"{state} --{action}--> {next_state}, "
            f"Reward = {reward}"
        )

print("\nINITIAL POLICY:", policy)


# Policy Iteration
iteration = 0

while True:
    iteration += 1
    print(f"\n========== POLICY ITERATION {iteration} ==========")

    # 1. Policy Evaluation
    evaluation_step = 0

    while True:
        delta = 0

        for state in states:
            old_value = V[state]
            next_state, reward = step(state, policy[state])

            V[state] = reward + gamma * V[next_state]
            delta = max(delta, abs(old_value - V[state]))

        evaluation_step += 1

        print(
            f"Evaluation {evaluation_step}: "
            f"S0 = {V['S0']:.6f}, "
            f"S1 = {V['S1']:.6f}, "
            f"G = {V['G']:.6f}, "
            f"Delta = {delta:.8f}"
        )

        if delta < theta:
            break

    # 2. Calculate action values Q(s, a)
    Q = {}

    for state in states:
        Q[state] = {}

        for action in actions:
            next_state, reward = step(state, action)
            Q[state][action] = (
                reward + gamma * V[next_state]
            )

    # 3. Policy Improvement
    new_policy = {}

    for state in states:
        best_action = max(
            actions,
            key=lambda action: Q[state][action]
        )
        new_policy[state] = best_action

    print("Updated policy:", new_policy)

    # 4. Check policy stability
    if new_policy == policy:
        print("Policy is stable. Algorithm stopped.")
        policy = new_policy
        break

    policy = new_policy


# Final results
print("\n========== FINAL RESULTS ==========")

print("\nOptimal State Values:")
for state in all_states:
    print(f"V*({state}) = {V[state]:.2f}")

print("\nFinal Action-Value Table Q*(s, a):")
print(f"{'State':<8}", end="")
for action in actions:
    print(f"{action:>12}", end="")
print()

for state in states:
    print(f"{state:<8}", end="")
    for action in actions:
        print(f"{Q[state][action]:>12.2f}", end="")
    print()

print("\nOptimal Policy:")
for state in states:
    print(f"Policy({state}) = {policy[state]}")

print("\nBellman Optimality Verification:")
for state in states:
    max_q = max(Q[state].values())
    print(
        f"V*({state}) = {V[state]:.2f}, "
        f"max Q*({state}, a) = {max_q:.2f}"
    )



STATES: ['S0', 'S1', 'G']
ACTIONS: ['up', 'down', 'left', 'right']
Discount Factor: 0.9
Normal Reward: -1
Wall Reward: -1

TRANSITION VERIFICATION
S0 up -> S0 Reward: -1
S0 down -> S0 Reward: -1
S0 left -> S0 Reward: -1
S0 right -> S1 Reward: -1
S1 up -> S1 Reward: -1
S1 down -> G Reward: 10
S1 left -> S0 Reward: -1
S1 right -> S1 Reward: -1

INITIAL VALUES: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

VALUE ITERATION
Iteration | V(S0) | V(S1) | Delta
1 | -1.0 | 10.0 | 10.0
2 | 8.0 | 10.0 | 9.0
3 | 8.0 | 10.0 | 0.0

FINAL OPTIMAL STATE VALUES
V*( S0 ) = 8.0
V*( S1 ) = 10.0
V*( G ) = 0.0

ACTION VALUES AND OPTIMAL POLICY

State: S0
Q( S0 , up ) = 6.2
Q( S0 , down ) = 6.2
Q( S0 , left ) = 6.2
Q( S0 , right ) = 8.0
Optimal action(s): ['right']

State: S1
Q( S1 , up ) = 8.0
Q( S1 , down ) = 10.0
Q( S1 , left ) = 6.2
Q( S1 , right ) = 8.0
Optimal action(s): ['down']

BELLMAN OPTIMALITY VERIFICATION
S0 V* = 8.0 max Q* = 8.0 Verified = True
S1 V* = 10.0 max Q* = 10.0 Verified = True

EXPLANATION
S1: Mov